# Development-only logistic-regression analysis

This notebook is the human-readable analysis layer for the aggregate `logistic_l2_balanced` development report. The reusable implementation remains in `training/`; this notebook does not duplicate model logic.

Scope and safeguards:

- 405-participant development partition only;
- three separate domain classifiers using the same folds and preprocessing contract;
- no participant identifiers, raw rows, or row-level predictions are displayed;
- the locked 102-participant final-test partition is not evaluated; and
- results are non-diagnostic research evidence, not final clinical performance.

> The operational predicted-probability threshold of 0.50 was reviewed and approved by the consultant pediatrician for this model version.

The repository versions that cutoff as `domain-probability-threshold-v1` for model configuration `logistic-l2-balanced-v1`. The approval does not automatically transfer to a different model or retraining cycle.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display


def find_repository_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'training').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from inside the Research_Component repository.')


ROOT = find_repository_root()
REPORT_PATH = ROOT / 'reports' / 'generated' / 'development-logistic-report.json'
if not REPORT_PATH.is_file():
    raise FileNotFoundError(
        'Aggregate report not found. Run: python -B -m training.run_development_experiment'
    )

report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
candidate_name = 'logistic_l2_balanced'
candidate = report['candidates'][candidate_name]
threshold = report['decision_threshold']

assert report['scope'] == 'development_only'
assert report['final_test_evaluated'] is False
assert threshold['value'] == 0.50
assert threshold['status'] == 'consultant_pediatrician_reviewed_and_approved'
assert threshold['version'] == 'domain-probability-threshold-v1'
assert candidate['model_configuration_version'] == 'logistic-l2-balanced-v1'

display(Markdown(f"**Loaded:** `{REPORT_PATH.relative_to(ROOT)}`"))

## Target and decision thresholds are different

- The observed follow-up domain score `<= 50` defines the training target.
- The model probability `>= 0.50` defines the predicted class for the current approved model configuration.

A score of 50 is not a 50% model probability. They operate at different stages.

In [ ]:
provenance = pd.DataFrame(
    [
        ('Experiment status', report['status']),
        ('Evaluation scope', report['scope']),
        ('Development participants', report['development_participants']),
        ('Final test evaluated', report['final_test_evaluated']),
        ('Model configuration', candidate['model_configuration_version']),
        ('Feature schema', report['feature_schema_version']),
        ('Target definition', report['target_definition_version']),
        ('Threshold version', threshold['version']),
        ('Threshold value', threshold['value']),
        ('Threshold status', threshold['status']),
        ('CV folds × repeats', f"{report['cross_validation']['n_splits']} × {report['cross_validation']['n_repeats']}"),
    ],
    columns=['Field', 'Value'],
)
display(provenance.style.hide(axis='index'))

In [ ]:
def nested_keys(value):
    if isinstance(value, dict):
        for key, child in value.items():
            yield key
            yield from nested_keys(child)
    elif isinstance(value, list):
        for child in value:
            yield from nested_keys(child)


raw_identifier_fields = {'participant_number', 'participant_key', 'baseline_assessment_id'}
present_identifier_fields = raw_identifier_fields.intersection(nested_keys(report))
assert not present_identifier_fields, f'Unsafe report fields: {sorted(present_identifier_fields)}'
display(Markdown('**Privacy check passed:** no raw identifier fields are present in the report.'))

## Development performance

These values use each participant's out-of-fold probabilities averaged across the five repeats. PR-AUC is emphasized because positive outcomes are uncommon. They are development estimates and must not be described as final-test performance.

In [ ]:
domain_labels = {
    'target_communication_concern': 'Communication',
    'target_cognitive_concern': 'Cognitive',
    'target_social_emotional_concern': 'Social-emotional',
}
performance_rows = []
for target_name, domain in candidate['domains'].items():
    metrics = domain['averaged_participant_oof_metrics']
    performance_rows.append(
        {
            'Domain': domain_labels[target_name],
            'Positive count': domain['positive_count'],
            'Sensitivity': metrics['sensitivity'],
            'Specificity': metrics['specificity'],
            'Precision': metrics['precision'],
            'F1': metrics['f1'],
            'Balanced accuracy': metrics['balanced_accuracy'],
            'PR-AUC': metrics['average_precision_pr_auc'],
            'ROC-AUC': metrics['roc_auc'],
            'Brier score': metrics['brier_score'],
        }
    )

performance = pd.DataFrame(performance_rows).set_index('Domain')
display(performance.style.format('{:.4f}', subset=performance.columns[1:]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
for axis, (target_name, domain) in zip(axes, candidate['domains'].items()):
    counts = domain['averaged_participant_oof_metrics']['confusion_matrix']
    matrix = [
        [counts['true_negative'], counts['false_positive']],
        [counts['false_negative'], counts['true_positive']],
    ]
    axis.imshow(matrix, cmap='Blues')
    for row in range(2):
        for column in range(2):
            axis.text(column, row, matrix[row][column], ha='center', va='center')
    axis.set_xticks([0, 1], labels=['Predicted 0', 'Predicted 1'])
    axis.set_yticks([0, 1], labels=['Observed 0', 'Observed 1'])
    axis.set_title(domain_labels[target_name])
fig.suptitle('Development out-of-fold confusion matrices at approved threshold 0.50')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharex=True, sharey=True, constrained_layout=True)
for axis, (target_name, domain) in zip(axes, candidate['domains'].items()):
    bins = pd.DataFrame(domain['calibration_bins'])
    axis.plot([0, 1], [0, 1], '--', color='grey', label='Ideal')
    axis.plot(
        bins['mean_predicted_probability'],
        bins['observed_positive_rate'],
        marker='o',
        label='Observed',
    )
    axis.set_title(domain_labels[target_name])
    axis.set_xlabel('Mean predicted probability')
    axis.grid(alpha=0.25)
axes[0].set_ylabel('Observed positive rate')
axes[-1].legend()
fig.suptitle('Aggregate development calibration')
plt.show()

## Explainability: coefficient stability

The charts show the largest mean absolute standardized logistic coefficients and their variation across fitted folds. Coefficients describe model associations, not causation or clinical importance. Checkpoint coefficients are relative to the `CP2M` reference category.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
for axis, (target_name, domain) in zip(axes, candidate['domains'].items()):
    coefficients = pd.DataFrame(domain['coefficient_stability'])
    coefficients['absolute_mean'] = coefficients['mean_coefficient'].abs()
    selected = coefficients.nlargest(10, 'absolute_mean').sort_values('mean_coefficient')
    axis.barh(
        selected['transformed_feature'],
        selected['mean_coefficient'],
        xerr=selected['standard_deviation'],
        alpha=0.8,
    )
    axis.axvline(0, color='black', linewidth=0.8)
    axis.set_title(domain_labels[target_name])
    axis.set_xlabel('Mean coefficient ± fold SD')
fig.suptitle('Largest coefficient magnitudes across development folds')
plt.show()

In [ ]:
fold_pr_auc = []
labels = []
for target_name, domain in candidate['domains'].items():
    labels.append(domain_labels[target_name])
    fold_pr_auc.append(
        [row['metrics']['average_precision_pr_auc'] for row in domain['fold_results']]
    )
plt.figure(figsize=(8, 4))
plt.boxplot(fold_pr_auc, tick_labels=labels, showmeans=True)
plt.ylabel('Fold PR-AUC')
plt.title('Variation across 25 development folds per domain')
plt.grid(axis='y', alpha=0.25)
plt.show()

## Optional in-memory recomputation

The notebook normally reads the reproducible aggregate JSON. To demonstrate the full computation without writing another report, set the flag below to `True`. The existing modules still enforce the frozen development partition and final-test boundary.

In [ ]:
RUN_DEVELOPMENT_EXPERIMENT_IN_MEMORY = False

if RUN_DEVELOPMENT_EXPERIMENT_IN_MEMORY:
    from training.development_data import load_development_dataset
    from training.evaluation import run_development_cross_validation

    development = load_development_dataset()
    in_memory_results = run_development_cross_validation(development.records)
    print(
        'Recomputed',
        in_memory_results['development_participants'],
        'development participants; no report file was written.',
    )
else:
    print('Using the existing aggregate report; no model was recomputed.')

## Interpretation boundary and next decision

This notebook supports development-model review. It does not select a final model, validate calibration, or authorize final-test evaluation. The next model candidate should be compared with the same feature projection and shared fold plan. If the selected model configuration changes, applicability of the approved `0.50` threshold must be confirmed rather than assumed.